# core

> `core` contains the `CloakDF` class - a pseudonymisation/anonymisation engine for consistent cross-table key replacement and column masking, with Fernet-encrypted storage of mappings.

In [ ]:
#| default_exp core

## Setup & data

We use the [Northwind dataset](https://github.com/neo4j-contrib/northwind-neo4j) — a classic sample database with customers, orders, and employees tables that share keys across them.

In [ ]:
#| eval: false
import httpx
from pathlib import Path

base = "https://raw.githubusercontent.com/neo4j-contrib/northwind-neo4j/refs/heads/master/data"
files = ["customers.csv", "orders.csv", "employees.csv"]
dest = Path("../data")
dest.mkdir(exist_ok=True)

for f in files:
    (dest/f).write_bytes(httpx.get(f"{base}/{f}").content)
    print(f"✓ {f}")

✓ customers.csv
✓ orders.csv


✓ employees.csv


In [ ]:
#| eval: false
import pandas as pd
for f in files:
    df = pd.read_csv(dest/f, on_bad_lines='skip')
    print(f"\n{f}: {df.shape}")
    print(df.columns.tolist())
    display(df.head(2))


customers.csv: (67, 11)
['customerID', 'companyName', 'contactName', 'contactTitle', 'address', 'city', 'region', 'postalCode', 'country', 'phone', 'fax']


,customerID,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,fax
0,ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Obere Str. 57,Berlin,NaN,12209,Germany,030-0074321,030-0076545
1,ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Avda. de la Constitución 2222,México D.F.,NaN,05021,Mexico,(5) 555-4729,(5) 555-3745



orders.csv: (654, 14)
['orderID', 'customerID', 'employeeID', 'orderDate', 'requiredDate', 'shippedDate', 'shipVia', 'freight', 'shipName', 'shipAddress', 'shipCity', 'shipRegion', 'shipPostalCode', 'shipCountry']


,orderID,customerID,employeeID,orderDate,requiredDate,shippedDate,shipVia,freight,shipName,shipAddress,shipCity,shipRegion,shipPostalCode,shipCountry
0,10248,VINET,5,1996-07-04 00:00:00.000,1996-08-01 00:00:00.000,1996-07-16 00:00:00.000,3,32.38,Vins et alcools Chevalier,59 rue de l'Abbaye,Reims,NaN,51100,France
1,10249,TOMSP,6,1996-07-05 00:00:00.000,1996-08-16 00:00:00.000,1996-07-10 00:00:00.000,1,11.61,Toms Spezialitäten,Luisenstr. 48,Münster,NaN,44087,Germany



employees.csv: (9, 18)
['employeeID', 'lastName', 'firstName', 'title', 'titleOfCourtesy', 'birthDate', 'hireDate', 'address', 'city', 'region', 'postalCode', 'country', 'homePhone', 'extension', 'photo', 'notes', 'reportsTo', 'photoPath']


,employeeID,lastName,firstName,title,titleOfCourtesy,birthDate,hireDate,address,city,region,postalCode,country,homePhone,extension,photo,notes,reportsTo,photoPath
0,1,Davolio,Nancy,Sales Representative,Ms.,1948-12-08 00:00:00.000,1992-05-01 00:00:00.000,507 - 20th Ave. E. Apt. 2A,Seattle,WA,98122,USA,(206) 555-9857,5467,0x151C2F00020000000D000E0014002100FFFFFFFF4269...,Education includes a BA in psychology from Col...,2.0,http://accweb/emmployees/davolio.bmp
1,2,Fuller,Andrew,"Vice President, Sales",Dr.,1952-02-19 00:00:00.000,1992-08-14 00:00:00.000,908 W. Capital Way,Tacoma,WA,98401,USA,(206) 555-9482,3457,0x151C2F00020000000D000E0014002100FFFFFFFF4269...,Andrew received his BTS commercial in 1974 and...,NaN,http://accweb/emmployees/fuller.bmp


## Table configuration

Each table is configured with **key groups** (`id1`, `id2`, ...) and a list of columns to **mask**. Key groups ensure consistent pseudonymisation across tables — e.g. `customerID` maps to the same UUID in both `customers` and `orders` because both use group `id1`.

In [ ]:
#| eval: false
tables = {
    'customers': {
        'id1': 'customerID',
        'mask': ['contactName', 'address', 'phone', 'companyName', 'fax', 'city']
    },
    'orders': {
        'id1': 'customerID',
        'id2': 'employeeID',
        'mask': ['shipName', 'shipAddress']
    },
    'employees': {
        'id2': 'employeeID',
        'mask': ['firstName', 'lastName', 'address', 'homePhone', 'birthDate', 'notes']
    },
}

## Design decisions

### Why UUID4 for key pseudonymisation?
UUID4 is generated from **cryptographically random** bits,  no timestamp, MAC address, or sequence baked in. The fake ID reveals nothing about the original. With 122 random bits, collision probability is astronomically small.

### Shared vault for masked columns
Masked columns (names, addresses, etc.) use a **flat shared dict** across all tables. If `"Maria Anders"` appears in both `customers` and `orders`, it gets the same hex token. This preserves referential consistency without leaking the original value.

### `setdefault` pattern
Both `key_maps` and `vault` use `dict.setdefault()` , a get-or-insert in one call. This ensures existing mappings are reused and new values are only generated for unseen originals:
```python
km = self.key_maps.setdefault(group, {})  # create or retrieve inner dict
km.setdefault(val, str(uuid.uuid4()))     # only insert if missing
```

### Encryption at rest with Fernet
The mappings (`key_maps` + `vault`) are the most sensitive artefact ; they're the keys to de-anonymise everything. We use **Fernet** (AES-128-CBC + HMAC) from the `cryptography` library. The encryption key is never stored on the `CloakDF` instance — it's passed in at `save()`/`load()` time, keeping the class stateless with respect to secrets.

## CloakDF class

In [ ]:
#| export
#| export
import uuid, json
from pathlib import Path
from cryptography.fernet import Fernet
import pandas as pd


class CloakDF:
    """Pseudonymise and mask DataFrames consistently across related tables."""
    def __init__(self, tables):
        self.tables,self.vault,self.key_maps = tables,{},{}

    def encode(self, name, df):
        """Pseudonymise keys and mask columns for table `name`, returning new DataFrame."""
        df = self._pseudonymise_keys(name, df)
        return self._mask_columns(name, df)

    def decode(self, name, df):
        """Reverse `encode` — restore original keys and masked values."""
        df = df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            inv = {v: k for k, v in self.key_maps[group].items()}
            df[col] = df[col].map(inv)
        inv_vault = {v: k for k, v in self.vault.items()}
        for col in tbl.get('mask', []):
            df[col] = df[col].map(inv_vault)
        return df

    def _remap(self, df, col, mapping, id_fn):
        """Replace `col` values via `mapping`, adding new entries with `id_fn`."""
        for val in df[col].dropna().unique():
            mapping.setdefault(str(val), id_fn())
        df[col] = df[col].astype(str).map(mapping)
        return df

    def _pseudonymise_keys(self, name, df):
        """Replace id columns with consistent UUIDs via `key_maps`."""
        df = df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            assert df[col].notna().all(), f"Null found in {col} for table {name}"
            assert pd.api.types.is_string_dtype(df[col]), f"Column {col} must be string type, got {df[col].dtype}"
            km = self.key_maps.setdefault(group, {})
            df = self._remap(df, col, km, lambda: str(uuid.uuid4()))
        return df

    def _mask_columns(self, name, df):
        """Replace mask columns with truncated hex tokens via shared `vault`."""
        tbl = self.tables[name]
        for col in tbl.get('mask', []):
            df = self._remap(df, col, self.vault, lambda: uuid.uuid4().hex[:12])
        return df

    def save(self, path, key):
        """Encrypt and save `key_maps` + `vault` to `path` using Fernet `key`."""
        data = json.dumps({'key_maps': self.key_maps, 'vault': self.vault})
        Path(path).write_bytes(Fernet(key).encrypt(data.encode()))

    @classmethod
    def load(cls, path, key, tables):
        """Load encrypted mappings from `path` and return a new `CloakDF` instance."""
        data = json.loads(Fernet(key).decrypt(Path(path).read_bytes()))
        clk = cls(tables)
        clk.key_maps, clk.vault = data['key_maps'], data['vault']
        return clk

    @staticmethod
    def generate_key(path=None):
        """Generate a Fernet key, optionally saving to `path`."""
        key = Fernet.generate_key()
        if path: Path(path).write_bytes(key)
        return key

    @staticmethod
    def load_key(path=None, env_var=None):
        """Load a Fernet key from `path` or `env_var`."""
        if path: return Path(path).read_bytes()
        if env_var:
            import os
            return os.environ[env_var].encode()
        raise ValueError("Provide either path or env_var")

## Round-trip test

Encode all three tables, save encrypted mappings, load from a fresh instance, decode, and verify we get the originals back.

In [ ]:
#| eval: false
import os

# 1. Generate key and store in env var
key = CloakDF.generate_key()
os.environ['CLOAKDF_KEY'] = key.decode()
loaded_key = CloakDF.load_key(env_var='CLOAKDF_KEY')
assert loaded_key == key
print("✓ Key round-tripped via env var")

# 2. Encode all tables
clk = CloakDF(tables)
originals, encoded = {}, {}
for name in tables:
    df = pd.read_csv(dest/f"{name}.csv", on_bad_lines='skip')
    for k, v in tables[name].items():
        if k.startswith('id'): df[v] = df[v].astype(str)
    originals[name] = df
    encoded[name] = clk.encode(name, originals[name])
    print(f"✓ Encoded {name}")

# 3. Save, load with env key, decode and compare
# --- Key loading options (uncomment one) 
# loaded_key = CloakDF.load_key(path=dest/"secret.key")
# loaded_key = CloakDF.load_key(env_var='CLOAKDF_KEY')
clk.save(dest/"mappings.enc", loaded_key)
clk2 = CloakDF.load(dest/"mappings.enc", loaded_key, tables)
for name in tables:
    decoded = clk2.decode(name, encoded[name])
    pd.testing.assert_frame_equal(decoded, originals[name])
    print(f"✓ Decoded {name} matches original")

✓ Key round-tripped via env var
✓ Encoded customers
✓ Encoded orders
✓ Encoded employees
✓ Decoded customers matches original
✓ Decoded orders matches original
✓ Decoded employees matches original


In [ ]:
#| eval: false
cols = ['employeeID', 'firstName', 'lastName', 'title', 'address', 'homePhone']
display(encoded['employees'][cols].head(3))
display(decoded[cols].head(3))

,employeeID,firstName,lastName,title,address,homePhone
0,dcdc9a63-bfd3-4028-8a50-72f0b890adc6,a75b1e84efb9,4ae3d7a62143,Sales Representative,71ac5533d30e,607c88feacc7
1,070391df-c952-4124-8ae3-e3f58e794b24,cef51a1d70b3,3c8fe1360470,"Vice President, Sales",5e309c8f9320,0747f93f0117
2,21395485-312a-45de-98d1-8d70f3a82506,3e1e149f1327,e1981fb61187,Sales Representative,58fe249ebced,ce5116345459


,employeeID,firstName,lastName,title,address,homePhone
0,1,Nancy,Davolio,Sales Representative,507 - 20th Ave. E. Apt. 2A,(206) 555-9857
1,2,Andrew,Fuller,"Vice President, Sales",908 W. Capital Way,(206) 555-9482
2,3,Janet,Leverling,Sales Representative,722 Moss Bay Blvd.,(206) 555-3412


on gihub pages the source is not rendering properly im gettings this error

https://github.com/giordafrancis/cloakdf/blob/main/cloakdf/core.py#L15

it's assuming code is on main and not master. How best to change this in the project?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

This is a common issue with nbdev-generated docs — by default it assumes the branch is `main`, but your repo uses `master`.

The fix is to set the `branch` field in your `settings.ini` file:

```ini
branch = master
```

Do you have a `settings.ini` in your project root? If so, check whether there's already a `branch` line in it that needs changing, or if it's missing entirely.

In [ ]:
!ls ..


LICENSE      README.md	cloakdf		  data	nbs
MANIFEST.in  _proc	cloakdf.egg-info  dist	pyproject.toml


no just py project.toml , woudl this be here

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes! In newer nbdev projects, the config lives in `pyproject.toml` under the `[tool.nbdev]` section. You'd add:

```toml
[tool.nbdev]
branch = master
```

Can you show what's in your `pyproject.toml`? That way we can see if there's already a `[tool.nbdev]` section to add it to.